In [1]:
from pinecone import RerankModel

# 4. Define your query & documents
query = "Tell me about Apple's products"
documents = [
    "An apple is a sweet, edible fruit produced by an apple tree.",
    "Apple Corporation designs and manufactures smartphones like iPhones and laptops like MacBooks.",
    "Oranges and bananas are other popular fruits, just like the apple.",
    "The tech giant Apple released a new version of iOS today.",
    "Fresh apples can be used to bake a delicious apple pie."
]

# 5. Call the reranker
reranked = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=query,
    documents=[{"id": str(i), "text": doc} for i, doc in enumerate(documents)],
    top_n=3
)

# 6. Inspect reranked results
def show_reranked_results(query, matches):
    print(f"Query: {query}")
    for i, m in enumerate(matches):
        print(f"{i+1}. Score: {m.score:.4f} | Text: {m.document.text}")

show_reranked_results(query, reranked.data)

ModuleNotFoundError: No module named 'pinecone'

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch

# Get cloud and region settings
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')

# Define serverless specifications
spec = ServerlessSpec(cloud=cloud, region=region)

# Define index name
index_name = 'medical-notes-index'

# Clean up any existing index with the same name
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)

# Create a new index
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

In [ ]:
import requests
import tempfile

with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, "sample_notes_data.jsonl")

    # Download the file from github
    url = "https://raw.githubusercontent.com/devtlv/Datasets-GEN-AI-Bootcamp/refs/heads/main/Week%208/W8D2/data.json"
    response = requests.get(url)
    response.raise_for_status()

    with open(file_path, "wb") as f:
        f.write(response.content)

    df = pd.read_json(file_path, orient='records', lines=True)

# Preview the DataFrame
print("Data shape:", df.shape)
df.head()

In [ ]:
# Instantiate an index client
index = pc.Index(name=index_name)

# Upsert data into index from DataFrame
index.upsert_from_dataframe(df)

# Wait for availability
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print(f"Vector count: ", vector_count)
    return vector_count > 0

while not is_fresh(index):
    time.sleep(5)

print("Index ready!")
index.describe_index_stats()

In [ ]:
# 1. Define your embedding function
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=1)
    return embedding

# 2. Run a semantic search query
question = "Patient has severe chest pain and shortness of breath"
query = get_embedding(question).tolist()

# Get results
results = index.query(vector=[query], top_k=5, include_metadata=True)

# Sort results by score in descending order
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

In [ ]:
# 1. Display initial search results
def show_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nResults:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match["id"]}')
        print(f' Score: {match["score"]}')
        print(f' Metadata: {match["metadata"]}')
        print('')

show_results(question, sorted_matches)

# 2. Prepare documents for reranking
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f"{key}: {value}" for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

# 3. Execute serverless reranking
refined_query = "What is the recommended treatment for acute chest pain?"
reranked_results = pc.inference.rerank(
    model="bge-reranker-v2-m3",
    query=refined_query,
    documents=transformed_documents,
    rank_fields=["reranking_field"],
    top_n=3,
    return_documents=True,
)

# 4. Show reranked results
def show_reranked_results(question, matches):
    print(f'Question: \'{question}\'')
    print('\nReranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.text}')
        print('')

show_reranked_results(refined_query, reranked_results.data)

# 5. Clean up (optional)
# pc.delete_index(name=index_name)